# ESA WorldCover — 10 m land cover

[ESA WorldCover](https://esa-worldcover.org/) is a global 10 m land-cover map
derived from Sentinel-1 and Sentinel-2. This notebook fetches the 2021 (v200)
tile for a small AOI over the Gulf of Guinea via the `amazon-s3` backend, then
reads the cropped GeoTIFF back with pyramids' `Dataset`.

## Setup

The imports: `tempfile` / `Path` for a scratch download directory, `earthlens`
for the unified `EarthLens` entry point, and pyramids' `Dataset` for reading the
resulting raster.

In [ ]:
import tempfile
from pathlib import Path

from pyramids.dataset import Dataset

from earthlens.core import EarthLens

## Fetch the land-cover tile

WorldCover is a file-writing backend, so we give it a scratch directory to write
into. A fresh `tempfile.mkdtemp()` keeps the download out of the working tree.

In [ ]:
out = Path(tempfile.mkdtemp(prefix='wc_'))

### Build the request

Construct the `EarthLens` request first — the `amazon-s3` data source, the
`esa-worldcover` dataset, the 2021 date window, and the AOI bounding box. Keeping
the construction on its own line makes each parameter easy to read and tweak.

In [ ]:
src = EarthLens(
    data_source="amazon-s3",
    start='2021-01-01',
    end='2021-01-01',
    aoi=[6.40, 0.40, 6.45, 0.45],
    dataset='esa-worldcover',
    path=str(out),
)

### Download

`download()` fetches the tile, crops it to the AOI, and returns the list of
written output paths.

In [ ]:
paths = src.download(progress_bar=False)
paths

## Read the result

Read the first written GeoTIFF back with pyramids' `Dataset` and inspect its
bounding box and shape to confirm the crop landed on the requested AOI.

In [ ]:
raster = paths[0]
ds = Dataset.read_file(raster)
print('bbox:', [round(x, 3) for x in ds.bbox], 'shape:', ds.shape)